# 1. IMPORT LIBRARY

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 2. LOAD DATASET

In [2]:
DATA_PATH = Path("../data/processed")
feature_A = pd.read_csv(DATA_PATH / "feature_A.csv")
feature_AB = pd.read_csv(DATA_PATH / "feature_AB.csv")

# 3. FEATURE A

pisah feature & label

In [3]:
X_A = feature_A.drop(columns=["user_id", "label", "source_folder"], errors="ignore")
X_AB = feature_AB.drop(columns=["user_id", "label", "source_folder"], errors="ignore")
y_A = feature_A["label"].astype(str)

encode label

In [4]:
encoder = LabelEncoder()
y_A = encoder.fit_transform(y_A)

train test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_A, y_A, test_size=0.2, random_state=42, stratify=y_A
)

training

In [6]:
rf_A = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf_A.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


prediction

In [7]:
y_pred = rf_A.predict(X_test)

evaluation

In [8]:
acc_A = accuracy_score(y_test, y_pred)

pre_A = precision_score(
    y_test,
    y_pred,
    average="macro"
)

rec_A = recall_score(
    y_test,
    y_pred,
    average="macro"
)

f1_A = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("Accuracy :", acc_A)
print("Precision:", pre_A)
print("Recall   :", rec_A)
print("F1 Score :", f1_A)

Accuracy : 0.988865692414753
Precision: 0.9830115455706379
Recall   : 0.9867776453458619
F1 Score : 0.9848769177642938


classif report

In [9]:
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

              precision    recall  f1-score   support

         bot       0.99      0.99      0.99      2179
       human       0.97      0.98      0.98       695

    accuracy                           0.99      2874
   macro avg       0.98      0.99      0.98      2874
weighted avg       0.99      0.99      0.99      2874



# 4. FEATURE A+B

pisahkah feature & label

In [10]:
X_AB = feature_AB.drop(columns=["user_id", "label", "source_folder"], errors="ignore")
y_AB = feature_AB["label"].astype(str)

encoder = LabelEncoder()
y_AB = encoder.fit_transform(y_AB)

train test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_AB,
    y_AB,
    test_size=0.2,
    random_state=42,
    stratify=y_AB
)

training

In [12]:
rf_AB = RandomForestClassifier(

    n_estimators=200,

    max_depth=None,

    random_state=42,

    n_jobs=-1
)

rf_AB.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


prediction

In [13]:
y_pred = rf_AB.predict(X_test)

evaluation

In [14]:
acc_AB = accuracy_score(y_test, y_pred)

pre_AB = precision_score(
    y_test,
    y_pred,
    average="macro"
)

rec_AB = recall_score(
    y_test,
    y_pred,
    average="macro"
)

f1_AB = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("Accuracy :", acc_AB)
print("Precision:", pre_AB)
print("Recall   :", rec_AB)
print("F1 Score :", f1_AB)

Accuracy : 0.9895615866388309
Precision: 0.98351039306279
Recall   : 0.9882164942667252
F1 Score : 0.9858358522801944


classif report

In [15]:
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

              precision    recall  f1-score   support

         bot       1.00      0.99      0.99      2179
       human       0.97      0.99      0.98       695

    accuracy                           0.99      2874
   macro avg       0.98      0.99      0.99      2874
weighted avg       0.99      0.99      0.99      2874



feature importance

In [16]:
importance = pd.DataFrame({
    "Feature": X_AB.columns,
    "Importance": rf_AB.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(15)

,Feature,Importance
4,favourites_count,0.260459
11,favorites_per_day,0.258021
10,tweets_per_day,0.114483
2,statuses_count,0.092875
21,duplicate_content_score,0.057317
9,follow_ratio,0.034283
7,geo_enabled,0.031289
0,followers_count,0.027649
12,listed_ratio,0.019526
8,account_age_days,0.015706


# 5. RESULT

In [ ]:
results = pd.DataFrame({
    "Feature": ["A", "A+B"],
    "Accuracy": [acc_A, acc_AB],
    "Precision": [pre_A, pre_AB],
    "Recall": [rec_A, rec_AB],
    "F1 Score": [f1_A, f1_AB]
})

results

,Feature,Accuracy,Precision,Recall,F1 Score
0,A,0.988866,0.983012,0.986778,0.984877
1,A+B,0.989562,0.983510,0.988216,0.985836


In [ ]:
RESULT_PATH = Path("../results")
RESULT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

results.to_csv(
    RESULT_PATH / "random_forest_results.csv",
    index=False
)

In [19]:
import joblib

MODEL_PATH = Path("../models")

MODEL_PATH.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(rf_A, MODEL_PATH / "rf_feature_A.pkl")
joblib.dump(rf_AB, MODEL_PATH / "rf_feature_AB.pkl")
joblib.dump(encoder, MODEL_PATH / "label_encoder.pkl")

['..\\models\\label_encoder.pkl']